# Week 3: The copilot, end to end

All six capabilities are imported from `src.llm.copilot`. The database has to be running. Each section makes model calls, so run the notebook from the top and leave it time to finish.

`numbers not in the facts` should be an empty list. A figure that shows up there was not in the evidence the model was given.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.copilot import (
    ask_database,
    ask_support,
    explain_churn,
    generate_report,
    investigate_churn,
    recommend_action,
)
from src.llm.explain_churn import numbers_outside_facts


## Ask the database

The model writes a SELECT. The guard checks it, Postgres runs it as the read-only role, and the model explains the rows.


In [2]:
for question in (
    "Which country has the most active subscribers?",
    "What is the overall churn rate?",
):
    result = ask_database(question)
    print(question)
    print("status:", result["status"])
    print("sql:", result.get("sql"))
    print(result["answer"])
    print()


Which country has the most active subscribers?
status: ok
sql: SELECT c.country, COUNT(DISTINCT s.customer_id) AS active_subscribers FROM customers AS c JOIN subscriptions AS s ON c.customer_id = s.customer_id WHERE s.subscription_status = 'Active' GROUP BY c.country ORDER BY active_subscribers DESC LIMIT 1
The United States has the most active subscribers with 1079.

What is the overall churn rate?
status: ok
sql: SELECT AVG(CASE WHEN churned THEN 1.0 ELSE 0.0 END) AS overall_churn_rate FROM churn_labels LIMIT 1000
The overall churn rate is 0.248875.



## Ask what customers said

The count at the end of the answer is added by the code. The second question is not in the tickets, so the answer should say the records do not cover it.


In [3]:
for question in (
    "what are customers saying about payment failures?",
    "what are customers saying about the Mars colony shuttle?",
):
    result = ask_support(question)
    print(question)
    print("status:", result["status"])
    print(result["answer"])
    print()


what are customers saying about payment failures?
status: error
The language model could not be reached (ClientError).

what are customers saying about the Mars colony shuttle?
status: error
The language model could not be reached (ClientError).



## Explain a churn score

CUST003961 is the highest score. CUST004227 is the lowest. The facts are printed first so the note can be checked against them.


In [4]:
for customer_id in ("CUST003961", "CUST004227"):
    result = explain_churn(customer_id)
    print(customer_id, result["status"], result["band"], result["churn_probability"])
    print(result["facts"])
    print(result["answer"])
    print("numbers not in the facts:", numbers_outside_facts(result["answer"], result["facts"]))
    print()


CUST003961 error HIGH RISK 0.9901
Customer CUST003961
Churn probability: 0.9901
Decision threshold: 0.46
Band: HIGH RISK
Current plan: Standard
Profile segment: Corporate Bundle
Model segment: Dormant at-risk
Tenure: 14 days

Risk factors (on the risky side of the typical customer who stayed):
- Failed-payment rate: 100.0% (typical non-churner: 6.7%). This customer is higher than typical. The model associates higher values with higher churn risk.
- Engagement score: 0.398 (typical non-churner: 0.552). This customer is lower than typical. The model associates lower values with higher churn risk.
- Login frequency (sessions per active month): 0.00 (typical non-churner: 1.50). This customer is lower than typical. The model associates lower values with higher churn risk.
- Tenure: 14 days (typical non-churner: 1419 days). This customer is lower than typical. The model associates lower values with higher churn risk.
- Lifetime value: 0.00 (typical non-churner: 201.37). This customer is lowe

## Recommend an action

The rule is chosen in code. The model only writes the note. CUST000108 is high risk and high value. CUST000004 is the standard nurture case.


In [5]:
for customer_id in ("CUST000108", "CUST000004"):
    result = recommend_action(customer_id)
    print(customer_id, result["status"])
    print(result["rule_fired"])
    print("final action:", result["action"])
    print(result["message"])
    print("numbers not in the facts:", numbers_outside_facts(result["message"], result["facts"]))
    print()


CUST000108 error
High risk and high value -> Immediate personalized outreach
final action: Immediate personalized outreach
The language model could not be reached (ClientError).
numbers not in the facts: []

CUST000004 error
Medium or low risk and low value -> Standard nurture emails
final action: Standard nurture emails
The language model could not be reached (ClientError).
numbers not in the facts: []



## Write a report

Part A is calculated. Part B is the model's reading of Part A. This cell rewrites the weekly and monthly files in `reports/`.


In [6]:
for period in ("weekly", "monthly"):
    result = generate_report(period)
    print(result["period"], result["status"], result["label"])
    print(result["path"])
    print("numbers not in the facts:", result["numbers_outside_facts"])
    print()


weekly error Week of 2026-05-25 to 2026-05-31
/Users/cloudanix/Sahil/Sakshi/Project/netflix-customer-intelligence-ai-copilot/reports/weekly_2026-05-31.md
numbers not in the facts: []

monthly error May 2026
/Users/cloudanix/Sahil/Sakshi/Project/netflix-customer-intelligence-ai-copilot/reports/monthly_2026-05.md
numbers not in the facts: []



## Investigate a region

India's May churn fell. The United States in April is the second case. Atlantis is not a country, so that call should stop before the model.


In [7]:
for region, month in (("India", "2026-05"), ("United States", "2026-04"), ("Atlantis", "2026-05")):
    result = investigate_churn(region, month)
    print(region, month, result["status"])
    print(result["answer"])
    print("numbers not in the evidence:", result["numbers_outside_facts"])
    print()


India 2026-05 error
The language model could not be reached (ClientError).
numbers not in the evidence: []

United States 2026-04 error
The language model could not be reached (ClientError).
numbers not in the evidence: []

Atlantis 2026-05 no_such_region
No such region: Atlantis.
numbers not in the evidence: []

